[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/compute_module1/blob/main/notebooks/lab_06_counted_and_clocked.ipynb)

# Lab 6: Counted and Clocked

*From* **A First Taste of Algorithms — and the Machines That Learn** *(The Curious Family Library). A computational lab: run the cells in order. A few labs end with a deliberately broken “Break it” cell — that one is meant to fail, and seeing how is the point.*

*Accompanies Chapter 6. Until now this book has counted and refused to clock. Today both instruments run side by side: the count predicts the shapes, the clock reveals the constants, and a plot — your first — shows the two sorts diverging exactly as the chapter promised. Two instruments arrive, each with its debt: the clock because constants are real, the plot because six chapters of tables have earned a picture.*

## 1. The plodder, counted

Insertion sort, exactly as hands do it — each arrival walks left past everything bigger:

In [ ]:
def insertion_sort_count(a):
    comparisons = 0
    for i in range(1, len(a)):
        arriving = a[i]
        j = i - 1
        while j >= 0 and a[j] > arriving:
            comparisons = comparisons + 1
            a[j + 1] = a[j]          # shift the bigger card right
            j = j - 1
        if j >= 0:
            comparisons = comparisons + 1   # the comparison that said "stop"
        a[j + 1] = arriving
    return comparisons

hand = [7, 3, 9, 1, 8]
print(insertion_sort_count(hand), hand)

One behavior to notice and remember: this sorter works **in place** — it rearranges the very list you hand it, the way your hands rearrange the actual cards (print `hand` after the call: it changed). Merge sort, by contrast, built a *new* sorted list and left its input alone. The difference seems cosmetic today; in §3 it becomes load-bearing. Check the count against your paper run from Fluency 1.1. Then the gait audit, worst case (reverse-sorted), in the house style:

In [ ]:
for n in [8, 32, 128, 512, 2048]:
    print(n, insertion_sort_count(list(range(n, 0, -1))))

Verify each row equals n(n − 1)/2 exactly, and read the gait: n quadruples each row — the count multiplies by *sixteen*. The n² signature, measured.

## 2. The race, counted

Retype merge sort and its step counter from Lab 3 (`merge`, `merge_sort_count`) — retyping is review — then race the two on the same worst-case inputs:

In [ ]:
for n in [8, 32, 128, 512, 2048]:
    ins = insertion_sort_count(list(range(n, 0, -1)))     # fresh worst case each —
    mrg = merge_sort_count(list(range(n, 0, -1)))[1]      # remember: insertion mutates
    print(n, "  insertion:", ins, "  merge:", mrg, "  ratio:", round(ins / mrg, 1))

Read the ratio column: about 1 at the top, about 90 at the bottom, roughly doubling with each doubling of n. In the counted currency the halver wins at *every* size on this table — even n = 8 — and the gap only opens. Hold that fact; the clock is about to complicate it.

## 3. The clock arrives — with a protocol

Python's stopwatch is `time.perf_counter()`: call it, do work, call it again, subtract. But a computer is a busy place — your timing shares the machine with everything else running — so single measurements lie. The professional protocol: **repeat many times, take the median** (the middle value when sorted — outliers from a busy moment fall off the ends).

In [ ]:
import time

def clock(sorter, items, repeats):
    times = []
    for _ in range(repeats):
        copy = list(items)
        start = time.perf_counter()
        sorter(copy)
        times.append(time.perf_counter() - start)
    times.sort()                      # the built-in sort; §6 turns the clock on it
    return times[len(times) // 2]     # the median

(Note the line `copy = list(items)`: each repeat sorts a **fresh copy**. For an in-place sorter this is not tidiness, it is the experiment's integrity — predict what the second repeat would be measuring without it, then hold the thought for Break-it 2.)

## 4. The clocked race, and the neighborhood

Now both sorts on the clock. Plain versions without counters (retype `merge_sort` from Lab 3; write `insertion_sort` by deleting the counter from §1), shuffled inputs this time — the clock's question is "typical day," and `random.shuffle` deals one:

In [ ]:
import random
random.seed(7)                        # same shuffles every run: experiments should repeat

for n in [4, 8, 16, 32, 64, 128, 256]:
    items = list(range(n))
    random.shuffle(items)
    t_ins = clock(insertion_sort, items, 51)
    t_mrg = clock(merge_sort, items, 51)
    winner = "insertion" if t_ins < t_mrg else "merge"
    print(n, "  insertion:", round(t_ins * 1e6, 1), "µs   merge:",
          round(t_mrg * 1e6, 1), "µs   →", winner)

Predict before running: §2 said merge wins every count — does it win every clock? Then read the table. On most machines a *neighborhood* appears at small n where insertion wins the clock while losing the count: the plodder is a simple creature with almost no overhead, and at a dozen items, overhead is most of the bill. Find your machine's crossover point (the n where the winner flips) and write it down — it is data for Problem 2.1, and it is the constant factor, located. The count predicted the shapes; the constants own the neighborhood.

## 5. The first plot

Six chapters of tables; time for the picture. `matplotlib` draws it — three moves for now: `plt.plot` draws a curve from lists of x and y values, `plt.loglog` does the same with both axes multiplying (so power laws become straight lines), and `plt.show()` reveals.

In [ ]:
import matplotlib.pyplot as plt

ns = [8, 32, 128, 512, 2048]
ins_counts = []
mrg_counts = []
for n in ns:
    worst = list(range(n, 0, -1))
    ins_counts.append(insertion_sort_count(worst)[1])
    mrg_counts.append(merge_sort_count(worst)[1])

plt.loglog(ns, ins_counts, "o-", label="insertion (counted)")
plt.loglog(ns, mrg_counts, "s-", label="merge (counted)")
plt.xlabel("n")
plt.ylabel("steps")
plt.legend()
plt.show()

Two nearly straight lines, diverging — and their *slopes* are the exponents: the insertion line climbs twice as steeply (n² against n·log n's almost-1). You are looking at Θ notation before it had a name: the plot ignores constants (they only shift a line up or down on these axes) and shows shape, which is exactly the deal the notation offers. Then plot the clocked data from §4 the same way and find the crossover as a *crossing of curves* — the picture of the constants' neighborhood.

## 6. The humbling

One more clock. Python ships with `sorted()` — Timsort, the deep water's engineered descendant of exactly the two algorithms on your screen. Predict the ratio before running: your merge sort versus the professionals, n = 2,048, shuffled. Twice as fast? Five times?

In [ ]:
items = list(range(2048))
random.shuffle(items)
t_mine = clock(merge_sort, items, 11)
t_builtin = clock(sorted, items, 11)
print("mine:", round(t_mine * 1e3, 2), "ms   built-in:",
      round(t_builtin * 1e3, 2), "ms   ratio:", round(t_mine / t_builtin))

The ratio is humbling — and the humility has *structure*, so spend it wisely. The built-in is the same Θ(n log n) as yours; the fence guarantees it cannot be a different shape. The entire margin is constants: decades of engineering, plus the quiet fact that `sorted` runs in C beneath Python's floor. Your algorithm and theirs will scale *identically* — double n and both double-and-a-bit — which you can verify by clocking both at 4,096 and checking the two ratios-to-before. The shapes are yours; the constants are an industry; and you now own the instrument that tells those two stories apart.

## Break it

Predict first. Run second. Explain any gap third.

**1.** Feed §2's counted race *already-sorted* inputs instead of reversed: `list(range(n))` in both places. Predict both columns before running. One algorithm's column collapses to almost nothing; which, why, and what is its exact new count? Connect to the chapter's recorded virtue — and to why "insertion sort is Θ(n²)" needed the words *worst case* attached.

**2.** Sabotage the timing protocol: change `clock` so all repeats share one list instead of sorting fresh copies (`copy = items` instead of `list(items)`). Predict what happens to insertion sort's clocked numbers and *why* — then run §4's race with the corrupted clock and watch the standings change. Which algorithm benefited from the corruption, and what does Break-it 1 have to do with it? (Measurement bugs that flatter one contestant are the most dangerous kind; you have just built one.)

**3.** Time a *single* run instead of a median of 51 (set repeats to 1) and run the n = 64 comparison ten times in a loop, printing the winner each time. Predict: same winner all ten times? Report the tally. The median protocol exists because of what you just witnessed; state the lesson in one sentence about believing any single measurement, anywhere, ever.

**4.** Build the chapter's counting sort for exam scores (whole numbers 0–100): 101 buckets, one pass in, read out in order. Count its touches at n = 1,000,000 (simulate with random scores) and clock it against `sorted` on the same data. Predict the winner first — you are betting a linear algorithm against the best-engineered n log n on Earth. Then betray it: change the keys from scores to `random.random()` decimals and explain, in the theorem-and-gate language of the chapter, why your algorithm doesn't just slow down but *stops making sense*.



<div style="page-break-after: always;"></div>